In [2]:
import numpy as np

# System parameters
d = 3       # State dimension
m = 1       # Control input dimension
T = 16      # Time steps per trajectory
N = 10000   # Number of trajectories
dt = 0.1   # Time step size

# Set random seed for reproducibility
np.random.seed(42)

# System matrices
A = np.array([[-0.1, 0.2, 0.0],
              [-0.2, -0.1, 0.3],
              [0.0, -0.3, -0.1]])
B = np.random.randn(d, m) * 0.1

# Nonlinear term
def N_func(x):
    x1, x2, x3 = x
    return np.array([
        x2 * x3,
        -x1 * x3,
        x1 * x2
    ])

# Allocate memory
trajectories = np.zeros((N, T, d))
controls = np.zeros((N, T - 1, m))

# Generate N trajectories
for i in range(N):
    x = np.random.randn(d) * 0.5  # Initial state
    trajectories[i, 0] = x
    for t in range(T - 1):
        u = np.random.randn(m) * 0.5  # Random control input
        x = x + dt * (A @ x + N_func(x) + B @ u)
        trajectories[i, t + 1] = x
        controls[i, t] = u

# Save to .npy files
np.save("fractional_system_trajectories.npy", trajectories)
np.save("fractional_system_inputs.npy", controls)

In [ ]:
import numpy as np
import casadi as ca

# System and simulation parameters
d = 3        # state dimension
m = 1        # control dimension
T = 16       # horizon (number of time steps, note: controls are T-1)
N = 10000    # number of trajectories / MPC problems
dt = 0.1    # time step size

# Load the previously saved trajectories and extract initial states
trajectories = np.load("fractional_system_trajectories.npy")
initial_states = trajectories[:, 0, :]  # shape: (N, d)

# System matrices
A = np.array([[-0.1,  0.2,  0.0],
              [-0.2, -0.1,  0.3],
              [ 0.0, -0.3, -0.1]])
B = np.random.randn(d, m) * 0.1  # using the same B as in simulation

# Define the nonlinear term N(x)
def N_func(x):
    # x is a CasADi SX or MX vector
    return ca.vertcat(x[1]*x[2],
                      -x[0]*x[2],
                      x[0]*x[1])

# Pre-allocate arrays to store Q, R and optimal control sequences.
Qs = np.zeros((N, d, d))
Rs = np.zeros((N, m, m))
optimal_controls = np.zeros((N, T - 1, m))  # each optimal control sequence will have T-1 steps

# For reproducibility of random cost matrices:
np.random.seed(42)

# Loop over each initial state and solve the MPC problem with nonlinear dynamics
for i in range(N):
    x0_val = initial_states[i]  # initial condition for the i-th problem

    # Generate a random symmetric positive-definite Q and positive-definite R.
    Mq = np.random.randn(d, d)
    Q = Mq.T @ Mq + 0.01 * np.eye(d)
    Mr = np.random.randn(m, m)
    R = Mr.T @ Mr + 0.01 * np.eye(m)
    
    Qs[i] = Q
    Rs[i] = R

    # Create a new optimization problem in CasADi
    opti = ca.Opti()

    X = opti.variable(d, T)
    U = opti.variable(m, T - 1)

    # Initial condition constraint
    opti.subject_to(X[:, 0] == x0_val)

    # Dynamics constraints (nonlinear dynamics):
    for k in range(T - 1):
        x_k = X[:, k]
        u_k = U[:, k]
        # Nonlinear update
        x_next = x_k + dt * (A @ x_k + N_func(x_k) + B @ u_k)
        opti.subject_to(X[:, k + 1] == x_next)

    # Define the cost function:
    cost = 0
    for k in range(T - 1):
        cost += ca.mtimes([X[:, k].T, Q, X[:, k]]) + ca.mtimes([U[:, k].T, R, U[:, k]])
    cost += ca.mtimes([X[:, T - 1].T, Q, X[:, T - 1]])
    opti.minimize(cost)

    # Set solver options
    opts = {"ipopt.print_level": 0, "print_time": False}
    opti.solver("ipopt", opts)

    # Solve the optimization problem
    try:
        sol = opti.solve()
        U_opt = sol.value(U)  # shape: (m, T-1)
        optimal_controls[i] = U_opt.T.reshape(T - 1, m)
    except RuntimeError:
        # If the solver fails, fill the control sequence with NaNs
        optimal_controls[i] = np.nan
        print(f"Warning: MPC problem at index {i} did not solve successfully.")

    if i % 1000 == 0:
        print(f"Processed {i} / {N}")

# Save the results
np.save("optimal_control_U.npy", optimal_controls)
np.save("LQR_Q.npy", Qs)
np.save("LQR_R.npy", Rs)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

Processed 0 / 10000
Processed 1000 / 10000
Processed 2000 / 10000
Processed 3000 / 10000
Processed 4000 / 10000
Processed 5000 / 10000
Processed 6000 / 10000
Processed 7000 / 10000
Processed 8000 / 10000
Processed 9000 / 10000


In [10]:
np.save("optimal_control_U.npy", optimal_controls)
np.save("LQR_Q.npy", Qs)
np.save("LQR_R.npy", Rs)